# 0.5 Adaptation Vulnerability Prep

This prep notebook builds a vulnerability adaptation scenario and writes a
scenario basin risk CSV that can be consumed by the simulation notebooks.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd

from sovereign.flood import build_masked_vulnerability_scenario_curves


In [2]:
# USER CONFIG
model = "wri"
scenario_name = "pubinf_urban_mask_50pct"
vulnerability_reduction = 0.50
mask_value = 1
mask_path = Path.cwd().parent / "outputs" / "flood" / "adaptation" / "test_mask.tif"

# Target sector setup
target_sector_label = "Public"
target_exposure_name = "inf_pub_capstock.tif"
paired_public_component_name = "pub_nres"


In [3]:
# Paths and baseline inputs
root = Path.cwd().parent
flood_dir = root / "inputs" / "flood" / "maps"
exposure_dir = root / "outputs" / "exposure"
risk_map_dir = root / "outputs" / "flood" / "risk" / "maps"
risk_basin_path = root / "outputs" / "flood" / "risk" / "basins" / f"risk_basins_m-{model}.csv"
vulnerability_path = root / "inputs" / "flood" / "vulnerability" / "jrc_depth_damage.csv"
basin_path = root / "outputs" / "boundaries" / "analysis_basins.gpkg"

scenario_root = root / "outputs" / "flood" / "adaptation" / "vulnerability" / scenario_name
scenario_map_dir = scenario_root / "maps"
scenario_basin_dir = scenario_root / "basins"
scenario_map_dir.mkdir(parents=True, exist_ok=True)
scenario_basin_dir.mkdir(parents=True, exist_ok=True)

flood_dic = {
    5: "UGA_wri-flood_RP5.tif",
    10: "UGA_wri-flood_RP10.tif",
    25: "UGA_wri-flood_RP25.tif",
    50: "UGA_wri-flood_RP50.tif",
    100: "UGA_wri-flood_RP100.tif",
    250: "UGA_wri-flood_RP250.tif",
    500: "UGA_wri-flood_RP500.tif",
    1000: "UGA_wri-flood_RP1000.tif",
}

target_exposure_path = exposure_dir / target_exposure_name
paired_public_baseline_paths = {
    rp: str(risk_map_dir / f"WRI_{rp}_{paired_public_component_name}_cap_damages.tif")
    for rp in flood_dic
}

risk_data = pd.read_csv(risk_basin_path)
risk_data = risk_data.iloc[:, 1:]
risk_data["AEP"] = 1 / risk_data["RP"]
risk_data["Pr_L_AEP"] = np.where(risk_data["Pr_L"] == 0, 0, 1 / risk_data["Pr_L"])
risk_data.reset_index(drop=True, inplace=True)

risk_data.head()


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
4,4,UGA.3_1,Arua,1.061033e+09,2.0,6160.324707,6160.324707,5,Public,0.2,0.5


In [4]:
# Vulnerability curves
vuln_df = pd.read_csv(vulnerability_path)
v_heights = vuln_df["flood_depth"].to_list()
v_inf = vuln_df["africa_infrastructure"].to_list()
v_inf_adapted = (pd.Series(v_inf) * (1 - vulnerability_reduction)).tolist()

baseline_damage_function = [v_heights, v_inf]
adapted_damage_function = [v_heights, v_inf_adapted]


In [5]:
# Build scenario products
_, scenario_risk_df, adapted_raster_paths = build_masked_vulnerability_scenario_curves(
    baseline_risk_df=risk_data,
    basin_path=str(basin_path),
    flood_map_lookup=flood_dic,
    flood_dir=str(flood_dir),
    target_exposure_path=str(target_exposure_path),
    mask_path=str(mask_path),
    baseline_damage_function=baseline_damage_function,
    adapted_damage_function=adapted_damage_function,
    target_sector_label=target_sector_label,
    output_dir=str(scenario_map_dir),
    adapted_component_name_template=f"WRI_{{rp}}_pub_inf_cap_damages.tif",
    combined_sector_name_template=f"WRI_{{rp}}_pub_cap_damages.tif",
    baseline_component_paths_by_rp=paired_public_baseline_paths,
    mask_value=mask_value,
)

scenario_basin_path = scenario_basin_dir / f"risk_basins_m-{model}.csv"
scenario_risk_df.to_csv(scenario_basin_path, index=False)

print("Scenario basin CSV written to:")
print(scenario_basin_path)
scenario_risk_df.head()


C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\sovereign-risk\lib\site-packages\rasterstats\io.py:328: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Scenario basin CSV written to:
E:\Projects\sovereign-risk-uga\outputs\flood\adaptation\vulnerability\pubinf_urban_mask_50pct\basins\risk_basins_m-wri.csv


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP,component_type,exposure_share
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5,vulnerability_adapted,1.0
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5,vulnerability_adapted,1.0
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5,vulnerability_adapted,1.0
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5,vulnerability_adapted,1.0
4,4,UGA.3_1,Arua,1.061033e+09,2.0,3080.162354,6160.324707,5,Public,0.2,0.5,vulnerability_adapted,1.0
